# Лабораторная работа 4. Скользящее среднее и ARIMA

**Цель.** Сравнить сглаживание, ARMA/ARIMA-подходы и обработку пропусков/выбросов во временном ряду.


## Ход работы

Ниже последовательно выполняются подготовка данных, построение графиков, расчёт признаков или моделей и краткая интерпретация результата. Случайные процедуры фиксируются через seed, чтобы результаты можно было повторить.


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)


## Настройка вычислений

Подключаются библиотеки, которые нужны для ARIMA и обработки особенностей ряда. Фиксация случайности делает результаты повторяемыми при последующих запусках ноутбука.


In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.tsa.api as smt

## Подготовка входных данных

В этом блоке считываются данные, с которыми дальше будет работать лабораторная. После загрузки важно проверить формат индекса, названия колонок и размерность ряда, потому что от этого зависит корректность всех следующих расчётов.


In [ ]:
# аналогичным образом загружаем данные о пассажирах
ecg = pd.read_csv('calm_p.csv')

# неподходящий формат данных приводим к тому, с которым Pandas может работать

df = ecg.set_index('Time').sort_index()

## Сглаживание и локальные характеристики

Скользящие и экспоненциальные оценки показывают, как меняется средний уровень и разброс во времени. Они уменьшают влияние случайного шума и помогают увидеть более устойчивую структуру ряда.


In [ ]:
df['MA_window_3'] = df['1'].rolling(window=3).mean()
df

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
fig = plt.figure(figsize=(20, 9))
plt.plot(df['1'], label='Original data')
plt.plot(df['MA_window_3'], label='Moving Average')
plt.legend(fontsize="20")
plt.title('ЭКГ')
plt.ylabel('mV', fontsize="20")
plt.xlabel('sec', fontsize="20")
plt.show()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
df['MA_window_6'] = df['1'].rolling(window=600).mean()
fig = plt.figure(figsize=(20, 9))
plt.plot(df['1'], label='Original data')
plt.plot(df['MA_window_6'], label='Moving Average')
plt.legend(fontsize="20")
plt.title('ЭКГ')
plt.ylabel('mV', fontsize="20")
plt.xlabel('s')
plt.show()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
'''
Отрисовка скользящего среднего по медиане для ts с 95% доверительным интервалом для стандартного отклонения.
:parameter
  :param ts: датасет
  :param window: кол-во семплов в окне - для скользящих средних
  :param plot_ma: bool - whether plot moving average
  :param plot_intervals: bool - whether plot upper and lower bounds
'''
def plot_ts(ts, plot_ma=True, plot_intervals=True, window=100,
            figsize=(15,5)):
    rolling_mean = ts.rolling(window=window).mean()
    rolling_std = ts.rolling(window=window).std()
    plt.figure(figsize=figsize)
    plt.title(ts.name)
    plt.plot(ts[window:], label='Реальные значения', color="black")
    if plot_ma:
        plt.plot(rolling_mean, 'g', label='MA'+str(window),
                 color="red")
    if plot_intervals:
        lower_bound = rolling_mean - (1.96 * rolling_std)
        upper_bound = rolling_mean + (1.96 * rolling_std)
    plt.fill_between(x=ts.index, y1=lower_bound, y2=upper_bound,
                     color='lightskyblue', alpha=0.4)
    plt.legend(loc='best')
    plt.grid(True)
    plt.show()

## Следующий этап расчёта

Этот блок продолжает основную цепочку ARIMA и обработки особенностей ряда: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
plot_ts(df["1"], window=100)
plot_ts(df["1"], window=10)

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
from statsmodels.tsa.arima.model import ARIMA

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
ecg_for_arma = df['1'].iloc[:2000].astype(float)
arma_model = ARIMA(ecg_for_arma, order=(2, 0, 1))
arma_model_fit = arma_model.fit()


## Следующий этап расчёта

Этот блок продолжает основную цепочку ARIMA и обработки особенностей ряда: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
print(arma_model_fit.summary())


## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
X = df['1'].iloc[:2500].astype(float).values
size = 2000
horizon = min(200, len(X) - size)
train, test = X[:size], X[size:size + horizon]
model = ARIMA(train, order=(2, 1, 0))
model_fit = model.fit()
arma_predictions = model_fit.forecast(steps=horizon)


## Первичная проверка

Здесь выводятся первые строки, размеры или базовая статистика. Такой просмотр нужен, чтобы убедиться, что данные загрузились без смещения колонок и что значения выглядят ожидаемо.


In [ ]:
print(len(train))
print(len(test))
print(len(arma_predictions))


## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
fig = plt.figure(figsize=(20, 9))
dur = len(arma_predictions)

plt.plot(range(len(train)), train, label='Train data')
plt.plot(range(size, size + dur), test[:dur], label='Test data')
plt.plot(range(size, size + dur), arma_predictions, label='ARMA forecast')
plt.legend(fontsize=20)
plt.title('ЭКГ')
plt.ylabel('mV', fontsize=20)
plt.xlabel('sample', fontsize=20)
plt.show()


## Подготовка входных данных

В этом блоке считываются данные, с которыми дальше будет работать лабораторная. После загрузки важно проверить формат индекса, названия колонок и размерность ряда, потому что от этого зависит корректность всех следующих расчётов.


In [ ]:
# аналогичным образом загружаем данные о пассажирах
passengers = pd.read_csv('passengers.csv')
# неподходящий формат данных приводим к тому, с которым Pandas может работать
passengers['Month'] = pd.to_datetime(passengers['Month'])
# также устанавливаем индекс и сортируем
pdf = passengers.set_index('Month').sort_index()

## Следующий этап расчёта

Этот блок продолжает основную цепочку ARIMA и обработки особенностей ряда: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
passengers

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
p_arma_model = ARIMA(pdf['Passengers'], order=(5,0,3))
p_arma_model_fit = p_arma_model.fit()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
from IPython.display import clear_output

def live_plot(data_dict, figsize=(7,5), title=''):
    clear_output(wait=True)
    plt.figure(figsize=figsize)
    for label,data in data_dict.items():
        plt.plot(data, label=label)
    plt.title(title)
    plt.grid(True)
    plt.xlabel('epoch')
    plt.legend(loc='center left')
    plt.show()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
series = pdf["Passengers"]
#series = df['1']

size = int(series.shape[0] * 0.5)
train, test = series[:size], series[size:len(series)]
history = [x for x in train.values]
predictions = []

val = {"true":[], "predict":[]}
for t in test.values:
    model = ARIMA(history, order=(5,1,0)).fit()
    output = model.forecast()
    yhat = output[0]
    predictions.append(yhat)
    val["predict"].append(yhat)
    val["true"].append(t)
    history.append(t)
    live_plot(val)

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
X = pdf['Passengers'].values
size = int(len(X) * 0.66)
train, test = X[0:size], X[size:len(X)]
history = [x for x in train]
arma_predictions = list()
# walk-forward validation
for t in range(len(test)):
    model = ARIMA(history, order=(5,1,0))
    model_fit = model.fit()
    output = model_fit.forecast()
    yhat = output[0]
    arma_predictions.append(yhat)
    obs = test[t]
    history.append(obs)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
fig = plt.figure(figsize=(20, 9))
dur = len(arma_predictions)

plt.plot([i for i in range(0, size)], train, label='Train data')
plt.plot([i for i in range(size - 1, size - 1 + dur)], test[:dur], label='Test data')
plt.plot([i for i in range(size - 1, size - 1 + dur)], arma_predictions, label='ARMA forecast')
plt.legend(fontsize="20")
plt.title('Passengers')
plt.ylabel('Passengers', fontsize="20")
plt.xlabel('Month', fontsize="20")
plt.show()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
arima_model = ARIMA(pdf['Passengers'], order=(10,2,10))
arima_model_fit = arima_model.fit()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
print(arima_model_fit.summary())

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
arima_residuals = pd.DataFrame(arima_model_fit.resid)
arima_residuals.plot()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
# forecast предсказывает только следующее значение
output = arima_model_fit.forecast()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
X = pdf['Passengers'].values
size = int(len(X) * 0.66)
train, test = X[0:size], X[size:len(X)]
history = [x for x in train]
arima_predictions = list()
# walk-forward validation
for t in range(len(test)):
    arima_model = ARIMA(history, order=(10,2,5))
    arima_model_fit = arima_model.fit()
    output = arima_model_fit.forecast()
    yhat = output[0]
    arima_predictions.append(yhat)
    obs = test[t]
    history.append(obs)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
fig = plt.figure(figsize=(20, 9))
plt.plot([i for i in range(0, size)], train, label='Train data')
plt.plot([i for i in range(size - 1, len(X) - 1)], test, label='Test data')
plt.plot([i for i in range(size - 1, len(X) - 1)], arima_predictions, label='ARIMA forecast')
plt.legend(fontsize="20")
plt.title('Airline passengers by month')
plt.ylabel('Total passengers', fontsize="20")
plt.xlabel('Month', fontsize="20")
plt.show()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
fig = plt.figure(figsize=(20, 9))

delta = []

for i in range(len(arima_predictions)):
    delta.append(arma_predictions[i] - arima_predictions[i])

plt.plot(delta, label='ARIMA(10,2,5) -  ARMA(5,0,3) delta forecast', linewidth=5)
plt.legend(fontsize="20")
plt.title('Airline passengers by month')
plt.ylabel('Total passengers', fontsize="20")
plt.xlabel('Month', fontsize="20")
plt.show()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
o_df = pdf.copy()

o_df["Passengers"][50:55] = np.nan

o_df.plot()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# Ваш код
o_df['Passengers'] = o_df['Passengers'].interpolate(method='linear')

o_df.plot(title="среднее из предыдущего и последующего")

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# Ваш код
o_df['Passengers'] = o_df['Passengers'].bfill()

o_df.plot(title='Последующее значение')

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# ваш код
median_val = o_df['Passengers'].median()
o_df['Passengers'] = o_df['Passengers'].fillna(median_val)

o_df.plot(title='медиана')

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# Ваш код
mean_val = o_df['Passengers'].mean()
o_df['Passengers'] = o_df['Passengers'].fillna(mean_val)

o_df.plot(title='среднее')

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
o_df["Passengers"][50:55] = np.nan

lin_df = o_df.interpolate(method="linear")

cub_df = o_df.interpolate(method="cubic")

pol_df = o_df.interpolate(method="polynomial", order=5)

zero_df = o_df.interpolate(method="zero")


fig = plt.figure(figsize=(20, 9))
layout = (3, 2)
plt.subplots_adjust(wspace=0.2, hspace=0.5)

original_ax = plt.subplot2grid(layout, (0, 0))
lin_ax = plt.subplot2grid(layout, (1, 0))
cub_ax = plt.subplot2grid(layout, (2, 1))

pol_ax = plt.subplot2grid(layout, (1, 1))
zero_ax = plt.subplot2grid(layout, (2, 0))

o_df.plot(title="С пропусками", ax = original_ax)
o_df.plot(title="С пропусками", ax = plt.subplot2grid(layout, (0, 1)))
lin_df.plot(title="Линейная интерполяция", ax = lin_ax)
cub_df.plot(title="Кубическая", ax = cub_ax)
pol_df.plot(title="Полиноминальная", ax = pol_ax)
zero_df.plot(title="zero", ax=zero_ax)

## Настройка вычислений

Подключаются библиотеки, которые нужны для ARIMA и обработки особенностей ряда. Фиксация случайности делает результаты повторяемыми при последующих запусках ноутбука.


In [ ]:
import seaborn as sns

## Следующий этап расчёта

Этот блок продолжает основную цепочку ARIMA и обработки особенностей ряда: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
pdf["Passengers"].hist(bins=30, color="black")

## Следующий этап расчёта

Этот блок продолжает основную цепочку ARIMA и обработки особенностей ряда: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
ecg["1"].hist(bins=100, color="black",)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
fig = plt.figure(figsize=(14, 9))

sns.boxplot(pdf.Passengers).set_title('Пассажиры')

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
fig = plt.figure(figsize=(14, 9))
ax = sns.boxplot(ecg["1"])
ax.set_title('ЭКГ')

## Настройка вычислений

Подключаются библиотеки, которые нужны для ARIMA и обработки особенностей ряда. Фиксация случайности делает результаты повторяемыми при последующих запусках ноутбука.


In [ ]:
from sklearn import preprocessing, svm

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
scaler = preprocessing.StandardScaler()
perc = 0.01

local_df = ecg["1"]

ts_scaled = scaler.fit_transform(local_df.values.reshape(-1,1))
model = svm.OneClassSVM(nu=perc, kernel="rbf", gamma=0.01)
model.fit(ts_scaled)
## dtf output
dtf_outliers =local_df.to_frame(name="ts")
dtf_outliers["index"] = range(len(local_df))
dtf_outliers["outlier"] = model.predict(ts_scaled)
dtf_outliers["outlier"] = dtf_outliers["outlier"].apply(lambda
                                                            x: 1 if x==-1 else 0)
## plot
fig, ax = plt.subplots(figsize=(20, 9))
ax.plot(dtf_outliers["index"], dtf_outliers["ts"],
        color="black")
ax.scatter(x=dtf_outliers[dtf_outliers["outlier"]==1]["index"],
           y=dtf_outliers[dtf_outliers["outlier"]==1]['ts'],
           color='red', linewidths=10)
plt.title(f"Found {sum(dtf_outliers['outlier']==1)} outliers", fontdict={'fontsize': 20})
ax.grid(True)
plt.show()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
outliers_idx = dtf_outliers[dtf_outliers["outlier"]==1].index

ts_clean = local_df.copy()
ts_clean.loc[outliers_idx] = np.nan
ts_clean = ts_clean.interpolate(method="linear")
ax = local_df.plot(figsize=(20, 9), color="red", alpha=0.5,
             title="Remove outliers", label="original", legend=True, linewidth=2)
ts_clean.plot(ax=ax, grid=True, color="black",
              label="interpolated", legend=True)
plt.legend(fontsize="20")
plt.show()

## Настройка вычислений

Подключаются библиотеки, которые нужны для ARIMA и обработки особенностей ряда. Фиксация случайности делает результаты повторяемыми при последующих запусках ноутбука.


In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)

# Генерируем нестационарный ряд
n = 200
trend = np.linspace(50, 250, n)  # Линейный тренд от 50 к 250
noise = np.random.normal(0, 15, n)  # Шум
random_walk = np.cumsum(np.random.normal(0, 5, n))  # Random walk

sales = trend + random_walk + noise

df = pd.DataFrame({'day': np.arange(1, n+1),'sales': sales})
df.set_index('day', inplace=True)
df.head(10)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# Постройте график исходного временного ряда
# Используйте figsize=(14, 5), добавьте label, title, xlabel, ylabel
# Ваш код
import matplotlib.pyplot as plt

# Постройте график исходного временного ряда
plt.figure(figsize=(14, 5))
plt.plot(df['sales'], label='Sales')
plt.title('Daily Sales Time Series')
plt.xlabel('Day')
plt.ylabel('Sales Volume')
plt.legend()
plt.grid(True)
plt.show()

## Проверка стационарности

Тест Дики-Фуллера используется как формальная проверка наличия единичного корня. Если ряд нестационарен, для моделей прогноза обычно нужны преобразования, разности или удаление тренда.


In [ ]:
# Ваш код
from statsmodels.tsa.stattools import adfuller

def check_stationarity(timeseries):
    print('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)

    if dfoutput['p-value'] < 0.05:
        print("Ряд стационарен (отвергаем H0)")
    else:
        print("Ряд нестационарен (не отвергаем H0)")

# Ваш код
check_stationarity(df['sales'])

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
#  Применить первое дифференцирование (d=1)
# Используйте df['sales'].diff() для получения разностей
# Ваш код
df['sales_diff'] = df['sales'].diff()

df_diff = df.dropna(subset=['sales_diff'])

# Визуализация разностей (опционально, но полезно)
plt.figure(figsize=(14, 5))
plt.plot(df_diff['sales_diff'], label='First Difference', color='green')
plt.title('First Difference of Sales')
plt.legend()
plt.grid(True)
plt.show()

## Следующий этап расчёта

Этот блок продолжает основную цепочку ARIMA и обработки особенностей ряда: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
# Используйте функцию check_stationarity()
# Ваш код
check_stationarity(df_diff['sales_diff'])

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
# Вариант 1 : начните с ARIMA(1,1,1)
# Вариант 2 : переберите несколько вариантов и выберите по AIC
# Выбирайте как душе угодно)

# Жду ваш код
import warnings
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

best_aic = np.inf
best_order = None
best_model = None

# Перебор небольших значений p и q, так как d=1 мы уже определили
for p in range(0, 6):
    for q in range(0, 4):
        try:
            model = ARIMA(df['sales'], order=(p, 1, q))
            model_fit = model.fit()
            if model_fit.aic < best_aic:
                best_aic = model_fit.aic
                best_order = (p, 1, q)
                best_model = model_fit
        except:
            continue

print(f"Best ARIMA Order: {best_order}")
print(f"Best AIC: {best_aic}")

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
# Ваш код
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Разделим данные на train и test
train_size = int(len(df) * 0.8)
train, test = df['sales'][:train_size], df['sales'][train_size:]

history = [x for x in train]
predictions = list()

# Walk-Forward Validation
for t in range(len(test)):
    # Обучаем модель на истории
    # Примечание: для скорости можно использовать best_order, найденный ранее
    model = ARIMA(history, order=best_order if best_order else (1,1,1))
    model_fit = model.fit()

    # Делаем прогноз на 1 шаг вперед
    output = model_fit.forecast()
    yhat = output[0]
    predictions.append(yhat)

    # Добавляем реальное значение в историю для следующего шага
    obs = test.iloc[t]
    history.append(obs)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# Ваш код
# Расчет метрик
mae = mean_absolute_error(test, predictions)
rmse = np.sqrt(mean_squared_error(test, predictions))

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')

# Визуализация
plt.figure(figsize=(14, 5))
plt.plot(test.index, test.values, label='Actual Sales')
plt.plot(test.index, predictions, label='Predicted Sales', color='red')
plt.title('Walk-Forward Forecast vs Actual')
plt.xlabel('Day')
plt.ylabel('Sales')
plt.legend()
plt.grid(True)
plt.show()

## Вывод

В работе показан полный путь от подготовки временного ряда до визуализации, расчёта признаков или построения модели. Основные результаты следует оценивать по графикам и метрикам, полученным при запуске ноутбука.
